In [7]:
from google.colab import files
uploaded = files.upload()

Saving processed_coursera_data.json to processed_coursera_data.json
Saving combined_dataset.json to combined_dataset.json
Saving edx_courses.json to edx_courses.json
Saving edx_degree_programs.json to edx_degree_programs.json
Saving edx_executive_education_paidstuff.json to edx_executive_education_paidstuff.json
Saving edx_programs.json to edx_programs.json
Saving combine_preprocessing.py to combine_preprocessing.py


In [8]:
import os
print(os.listdir())

['.config', 'edx_degree_programs.json', 'processed_coursera_data.json', 'combine_preprocessing.py', 'edx_programs.json', 'edx_executive_education_paidstuff.json', 'combined_dataset.json', 'edx_courses.json', 'sample_data']


In [2]:
import os
print(os.listdir())

['.config', 'sample_data']


In [3]:
from google.colab import files
uploaded = files.upload()

Saving archive (2).zip to archive (2).zip


In [4]:
import zipfile

with zipfile.ZipFile("archive (2).zip", 'r') as zip_ref:
    zip_ref.extractall()

In [5]:
import os
print(os.listdir())

['.config', 'edx_degree_programs.json', 'processed_coursera_data.json', 'combine_preprocessing.py', 'archive (2).zip', 'edx_programs.json', 'edx_executive_education_paidstuff.json', 'combined_dataset.json', 'edx_courses.json', 'sample_data']


In [6]:
import pandas as pd

df = pd.read_json("combined_dataset.json")
print(df.columns)

Index(['url', 'type', 'course_name', 'organization', 'instructor', 'rating',
       'nu_reviews', 'description', 'skills', 'level', 'Duration', 'reviews',
       'total_assignment', 'total_app', 'total_programming', 'total_reading',
       'total_plugin', 'total_ungraded', 'total_quiz', 'total_teammate',
       'total_peer', 'total_discussion', 'total_video', 'has_assignment',
       'has_app', 'has_programming', 'has_reading', 'has_plugin',
       'has_ungraded', 'has_quiz', 'has_teammate', 'has_peer',
       'has_discussion', 'has_video', 'has_no_enrol', 'enrollments',
       'has_rating', 'subject', 'has_subject', 'provider'],
      dtype='object')


In [7]:
import pandas as pd

df = pd.read_json("combined_dataset.json")

# Use correct column
df = df[['course_name']]
df = df.dropna()

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['course_name'])

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [9]:
def recommend(course_title, top_n=5):
    if course_title not in df['course_name'].values:
        return "Course not found"

    idx = df[df['course_name'] == course_title].index[0]
    scores = list(enumerate(cosine_sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:top_n+1]

    return df['course_name'].iloc[[i[0] for i in scores]]

In [10]:
print(recommend(df['course_name'].iloc[0]))

12819       كيفية إنشاء صورة مصغرة لليوتيوب باستخدام كانفا
1772     Building Rust AWS Lambda Microservices with Ca...
12336    Where, Why, and How of Lambda Functions in Pyt...
7862                          Lambda Expressions with Java
8042                        Les Expressions Lambda et Java
Name: course_name, dtype: object


In [11]:
df = pd.read_json("combined_dataset.json")

# Combine features
df['content'] = df['course_name'] + " " + df['description']
df = df[['content', 'course_name']].dropna()

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['content'])

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [13]:
def recommend(course_title, top_n=5):
    idx = df[df['course_name'] == course_title].index[0]
    scores = list(enumerate(cosine_sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    return df['course_name'].iloc[[i[0] for i in scores]]

In [15]:
def recommend(course_title, top_n=5):
    # find matching titles (partial match)
    matches = df[df['course_name'].str.contains(course_title, case=False, na=False)]

    if matches.empty:
        return "Course not found"

    idx = matches.index[0]

    scores = list(enumerate(cosine_sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:top_n+1]

    return df['course_name'].iloc[[i[0] for i in scores]]

In [16]:
print(recommend("machine learning"))

7900           Launching into Machine Learning en Français
8322         Machine Learning in the Enterprise - Français
10900    Smart Analytics, Machine Learning, and AI on G...
4824                       Feature Engineering en Français
1136       Art and Science of Machine Learning en Français
Name: course_name, dtype: object
